# Colab Workshop: Ollama + RAG on 2024 Species Discoveries

This notebook is designed for **Google Colab** and has three sections:
1. Setup Ollama
2. Intro to RAG with **post-2023** Natural History Museum content
3. Advanced retrieval over the same 2024 discoveries

## How to Enable the Free GPU

Ask your students to do the following as their very first step:

- Open the Menu: Go to Runtime in the top navigation bar.
- Change Type: Select Change runtime type.
- Select GPU: Under the Hardware accelerator dropdown, choose T4 GPU (this is the standard free tier GPU).
- Save: Click Save. The notebook will briefly reconnect to a new virtual machine equipped with the GPU.

## How To Use This Notebook

Run the notebook from top to bottom in a fresh Colab runtime.

Expected timing on free Colab:
- Setup and model pull: 5 to 20 minutes
- Index build and first query: 2 to 8 minutes

If a runtime disconnects, restart and rerun all cells in order.

## 1) Setup Ollama

### What This Section Does

In this section you will:
- Install required Python packages
- Install and start the Ollama server in Colab
- Pull one local LLM and one embedding model
- Run a quick smoke test to confirm local inference works

After this section, you should be able to run LLM calls without any external API key.

### 1.1 Install Python Dependencies

These packages provide the RAG framework and Ollama integrations.
Run once per fresh Colab runtime.

In [ ]:
# Install Python dependencies for LlamaIndex + Ollama integrations
%pip install -q jedi llama-index llama-index-llms-ollama llama-index-embeddings-ollama

### 1.2 Install The Ollama Runtime

This installs the Ollama binary so Colab can host a local inference server.

In [ ]:
# Install required system dependency and Ollama binary in Colab
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

### 1.3 Start The Local Ollama Service

This launches Ollama at `127.0.0.1:11434`, which the later LlamaIndex cells will call.

In [ ]:
# Start Ollama server (safe to rerun)
import os
import time
import subprocess

os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

check = subprocess.run(["bash", "-lc", "pgrep -f 'ollama serve'"], capture_output=True, text=True)
if check.returncode != 0:
    _ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(4)
    print("Started Ollama server.")
else:
    print("Ollama server is already running.")

!ollama list

### 1.4 Pull Models

We pull two models:
- `llama3.2:3b` for generation
- `nomic-embed-text` for embeddings

This keeps the full pipeline local with no API keys.

In [ ]:
# Pull lightweight models for Colab
!ollama pull llama3.2:3b
!ollama pull nomic-embed-text

In [ ]:
# Quick smoke test
!ollama run llama3.2:3b "Hello! Anyone home?"

### Setup Checkpoint

If the smoke test responded correctly, your local model is ready.

If not:
- Re-run the server start cell
- Re-run the model pull cell
- Confirm `ollama list` shows `llama3.2:3b` and `nomic-embed-text`

## 2) Intro to RAG

### RAG Concept In One Minute

RAG has two core steps:
1. Retrieval: find relevant chunks from your documents
2. Generation: use those chunks to produce an answer

This reduces hallucination by grounding answers in retrieved source text.

### 2.1 Configure LLM And Embedding Model

Here we configure the two core components of RAG:
- an LLM to generate answers
- an embedding model to map document chunks into vector space

In [ ]:
# Configure LlamaIndex to use local Ollama models
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model="llama3.2:3b", request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print("LLM and embedding model configured.")

### 2.2 Load 2024 Discovery Source Pages

This step downloads Natural History Museum **2024 discovery pages** and converts them into plain text files for indexing.
It includes the *Myloplus sauron* story and the broader 2024 new-species roundup.

In [ ]:
# Download NHM 2024 source pages and save as clean text files
import os
import re
import html
import urllib.request

os.makedirs("data", exist_ok=True)

source_urls = {
    "sauron_piranha": "https://www.nhm.ac.uk/discover/news/2024/june/new-species-vegetarian-piranha-named-after-lord-of-the-rings-villain-sauron.html",
    "new_species_2024": "https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-describe-190-new-species-2024.html",
}

source_files = []
for slug, url in source_urls.items():
    out_path = f"data/{slug}.txt"
    source_files.append(out_path)
    if os.path.exists(out_path):
        print(f"Already prepared: {out_path}")
        continue

    with urllib.request.urlopen(url) as resp:
        raw_html = resp.read().decode("utf-8", "ignore")

    # Lightweight HTML-to-text cleanup for robust local indexing.
    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", raw_html)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    text = re.sub(r"(?is)<[^>]+>", " ", text)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"SOURCE_URL: {url}\n\n")
        f.write(text)

    print(f"Saved cleaned source text to {out_path}")

print("Prepared source files:")
for path in source_files:
    print("-", path)

### 2.3 Zero-Shot Test Before RAG Indexing

Ask the local model about *Myloplus sauron* **before** building any vector index.
This demonstrates how non-retrieval answers can be weak or uncertain for recent facts.

In [ ]:
# Zero-shot check (no retrieval context yet)
zero_shot_prompt = "What is a Myloplus sauron and why was it named that?"
zero_shot_response = Settings.llm.complete(zero_shot_prompt)

print("=== Zero-shot response (before indexing) ===")
print(getattr(zero_shot_response, "text", str(zero_shot_response)))

### 2.4 Build The Vector Index (Embedding Step)

During indexing, the embedding model converts the NHM document chunks into vectors.
These vectors are what retrieval uses to find relevant context.

In [ ]:
# Build a basic RAG index from NHM 2024 source files
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(input_files=source_files).load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=4)

print(f"Loaded {len(documents)} document(s) and built index.")

### 2.5 LLM Output Without Retrieval (After Index Build)

This step still calls the Ollama LLM directly with **no retrieval step**.
Compare this with the RAG answer in the next cell to see grounding improvements.

In [ ]:
# Direct LLM generation (no retrieval, no vector index lookup)
plain_prompt = "Describe the vegetarian piranha discovered in 2024 and the inspiration for its name."
plain_response = Settings.llm.complete(plain_prompt)

# LlamaIndex LLM wrappers may return either a string-like object or an object with .text
print(getattr(plain_response, "text", str(plain_response)))

### 2.6 Run Diverse RAG Questions

These example queries cover multiple 2024 species discoveries so you can better demonstrate retrieval breadth, not just one fact.

In [ ]:
# Intro RAG: run a diverse question set
sample_questions = [
    "Describe the vegetarian piranha discovered in 2024 and the inspiration for its name.",
    "Summarize the 2024 NHM new species announcement and give two notable examples.",
    "What does the 2024 species roundup suggest about how naming choices connect science and culture?",
]

for i, q in enumerate(sample_questions, 1):
    print(f"\n=== Question {i} ===")
    print(q)
    response = query_engine.query(q)
    print(response)

## 3) Advanced Retrieval

### Why Advanced Retrieval Matters

A basic retriever may return partially relevant chunks.
Reranking adds an extra scoring step to keep the most relevant context before final answer generation.

In practice, this often improves factual precision and relevance.

### 3.1 Baseline Retrieval

Start with standard vector retrieval so you can compare quality before adding reranking.

In [ ]:
# Baseline retrieval
query = "Compare at least two different 2024 species mentioned by NHM and explain why each stood out."
baseline_engine = index.as_query_engine(similarity_top_k=6)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

### 3.2 Add LLM Reranking

Reranking reviews the initially retrieved chunks and keeps only the most relevant ones for answer synthesis.

In [ ]:
# Advanced retrieval: LLM-based reranking
from llama_index.core.postprocessor import LLMRerank

reranker = LLMRerank(choice_batch_size=4, top_n=3)
rerank_engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[reranker],
)
rerank_response = rerank_engine.query(query)

print("=== Reranked response ===")
print(rerank_response)

In [ ]:
# Compare retrieved source chunks
print("--- Baseline source nodes ---")
for i, node in enumerate(baseline_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

print()
print("--- Reranked source nodes ---")
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

### Interpreting The Source Nodes

When comparing baseline vs reranked source nodes, look for:
- Higher relevance to the query intent
- Less off-topic context
- Better support for key claims in the final answer

## 4) Intent Routing: Q&A vs Summarize

In this section, we add a lightweight **router** in front of the RAG pipeline.
The router inspects the user request and selects one of two paths:

- `q&a`: Retrieve evidence and answer the question directly.
- `summarize`: Retrieve a broader set of evidence and generate a concise summary.

### Why This Matters

Routing is useful when one assistant needs to support multiple interaction styles without forcing the user to choose manually.
It also mirrors production agent patterns where a classifier or policy decides which tool/chain to run.

### How This Demo Router Works

1. Read the user prompt.
2. Detect summary intent from keywords such as `summarize`, `summary`, `overview`, `brief`, or `key points`.
3. If summary intent is detected, run a summarization prompt over retrieved context.
4. Otherwise, run normal reranked RAG Q&A.

Try prompts like:
- `Summarize the key species highlights from the 2024 NHM announcement.`
- `What is Myloplus sauron and why was it named that?`
- `Give me a brief overview of notable 2024 discoveries and why they matter.`

In [ ]:
# Section 4 routing demo: choose between Q&A and summarize
def route_intent(user_text: str) -> str:
    text = user_text.lower()
    summarize_keywords = ["summarize", "summary", "overview", "brief", "key points"]
    if any(k in text for k in summarize_keywords):
        return "summarize"
    return "q&a"

def run_routed_query(user_text: str):
    route = route_intent(user_text)
    print(f"Route selected: {route}")

    if route == "q&a":
        return rerank_engine.query(user_text)

    retriever = index.as_retriever(similarity_top_k=8)
    nodes = retriever.retrieve(user_text)
    context = "\n\n".join([n.node.get_content() for n in nodes])
    prompt = (
        "Summarize the following retrieved context in 5-8 bullet points. "
        "Focus on species names, notable traits, and why discoveries matter.\n\n"
        f"Context:\n{context}"
    )
    result = Settings.llm.complete(prompt)
    return getattr(result, "text", str(result))

your_question = input("Ask a NHM 2024 question or request a summary: ")
print(run_routed_query(your_question))

### Suggested Exercises

Try these quick experiments:
- Change `similarity_top_k` from 4 to 8 in intro RAG
- Change reranker `top_n` from 3 to 2 or 5
- Ask multi-part questions and observe whether reranking helps